In [ ]:
## Clone the Github Repo - dev branch

import shutil
import os, sys
import json

# Remove old copy if it exists
shutil.rmtree('/content/cen_sierra_pywr_new', ignore_errors=True)

# Clone the dev branch
!git clone --branch dev --single-branch https://github.com/Maburidi/cen_sierra_pywr_new.git

# Add repo to Python path
sys.path.insert(0, '/content/cen_sierra_pywr_new/')

# Move into repo
%cd /content/cen_sierra_pywr_new/

# Verify current branch
!git branch --show-current

In [ ]:
# Install Pywr

!bash install_dependencies.sh

In [ ]:
# Run the Models

################## RESULTS ######################

# A folder called "results" will be created, where the results will be stored. Similar thing for loggs

################# Arguments #####################

# -b: Basin of choice, ['stanislaus', 'tuolumne', 'merced', 'upper_san_joaquin']
# -s : starting year: gcms: 2006  , loca2_gcms: 2015      Livenh: 1990
# -e : end year:      gcms: 2099  , loca2_gcms: 2099      Livenh: 2012
# -p: include planning: [0,1], 1 for stanislaus and upper_san_joaquin, 0 for the rest
# -n: runname
# -d: Debug mode: [0,1], 1 to see more details for development
# -sc: Scenario set. ['debug','natural_flow', 'gcms', 'loca2_gcms']
# -gm: gcms models, if its   ['CCSM4_rcp85'(only tuolumne),'ACCESS1_0_rcp85]
# -lgm: LOCA2 gcms models    [ACCESS_CM2, ]

#======= CL arguments for Planning mode ---
# -m: Planning months, default 8,
# -bl: Number of piecewise blocks, default 5




################# RUN SCENARIOS ###################

#Examples: uncomment the command line to run the scenario you prefer:

#-----------------
#[A] LOCA2 GCMS SSP370 Dataset
#!python main.py -b 'stanislaus' -s 2045 -e 2074 -d 0 -p 1 -sc 'loca2_gcms' -lgm 'ACCESS_CM2' -m 2 -bl 2
!python main.py -b 'tuolumne' -s 2045 -e 2074 -d 0 -sc 'loca2_gcms' -lgm 'ACCESS_CM2'
#!python main.py -b 'upper_san_joaquin' -s 2045 -e 2074 -d 0 -p 1 -sc 'loca2_gcms' -lgm 'ACCESS_CM2' -m 2 -bl 2
!python main.py -b 'merced' -s 2045 -e 2074 -d 0 -sc 'loca2_gcms' -lgm 'ACCESS_CM2'

#-----------------
#[B] GCMS RCP85  Dataset
#!python main.py -b 'stanislaus' -s 2045 -e 2074 -d 0 -p 1 -sc 'gcms' -gm 'ACCESS1_0_rcp85' -m 2 -bl 2
!python main.py -b 'tuolumne' -s 2045 -e 2074 -d 0 -sc 'gcms' -gm 'ACCESS1_0_rcp85'
#!python main.py -b 'upper_san_joaquin' -s 2045 -e 2074 -d 0 -p 1 -sc 'gcms' -gm 'ACCESS1_0_rcp85' -m 2 -bl 2
!python main.py -b 'merced' -s 2045 -e 2074 -d 0 -sc 'gcms' -gm 'ACCESS1_0_rcp85'

#-----------------
#[C] Livenh Historical natural_flow
!python main.py -b 'tuolumne' -s 1990 -e 2012 -d 0 -sc 'natural_flow'
!python main.py -b 'merced' -s 1990 -e 2012 -d 0 -sc 'natural_flow'
#!python main.py -b 'stanislaus' -s 1990 -e 2012 -d 0 -p 1 -sc 'natural_flow'
#!python main.py -b 'upper_san_joaquin' -s 1990 -e 2012 -d 0 -p 1 -sc 'natural_flow'

# Climate Robustness of FlowPywr Operating Rules: Hydrology-Conditioned Performance

This notebook implements the first extension of the climate robustness study. It separates the analysis into three layers and adds a hydrology-conditioned comparison that matches future water years to historical Livneh water years with similar annual runoff, April-July runoff, runoff timing, and initial terminal reservoir storage.

The notebook is designed for Google Colab with the repository cloned at `/content/cen_sierra_pywr_new`. It is postprocessing only: it reads existing FlowPywr result CSVs and does not modify or rerun the Pywr models.

## 1. Purpose

Raw scenario comparisons show how model outputs differ across historical, GCM, and LOCA2 forcings. Those raw differences combine hydrologic stress with operating behavior. This notebook adds a matched-hydrology comparison: each future water year is compared with the most hydrologically similar historical Livneh water years in the same basin.

The analysis is organized into three layers:

- **Layer A: Hydrologic stress** describes what water availability, timing, and initial storage conditions the system experienced.
- **Layer B: Raw system outcomes** describes modeled storage, hydropower, deliveries, IFR performance, outflow, and flood-release proxy outcomes.
- **Layer C: Hydrology-conditioned performance** compares future outcomes against historical years with similar hydrologic stress.

## 2. Imports and Configuration

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)


def find_project_root():
    """Find the repo root from Colab, local Jupyter, or a subdirectory."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    candidates.extend([
        Path("/content/cen_sierra_pywr_new"),
        Path("/content/cen-sierra-pywr"),
    ])

    for candidate in candidates:
        if (candidate / "main.py").exists() and (candidate / "data").exists():
            return candidate

    content = Path("/content")
    if content.exists():
        for candidate in content.glob("*/"):
            if (candidate / "main.py").exists() and (candidate / "data").exists():
                return candidate

    raise FileNotFoundError(
        "Could not find the repo root. In Colab, run: %cd /content/cen_sierra_pywr_new"
    )


PROJECT_ROOT = find_project_root()
RESULTS_ROOT = next((p for p in [PROJECT_ROOT / "results", PROJECT_ROOT / "RESULTS_"] if p.exists()), None)
if RESULTS_ROOT is None:
    raise FileNotFoundError(f"Could not find results or RESULTS_ under repo root: {PROJECT_ROOT}")

OUTPUT_DIR = RESULTS_ROOT / "climate_robustness" / "hydrology_conditioned"
FIG_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

K_MATCHES = 3
HISTORICAL_SCENARIO = "Historical Livneh"
FUTURE_SCENARIOS = ["GCM ACCESS1-0 RCP8.5", "LOCA2 ACCESS-CM2"]

PALETTE = {
    "Historical Livneh": "#2F6B4F",
    "GCM ACCESS1-0 RCP8.5": "#B65D2E",
    "LOCA2 ACCESS-CM2": "#3C6EAA",
}

MARKERS = {
    "Historical Livneh": "o",
    "GCM ACCESS1-0 RCP8.5": "s",
    "LOCA2 ACCESS-CM2": "^",
}

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8,
})

print(f"Project root: {PROJECT_ROOT}")
print(f"Results root: {RESULTS_ROOT}")
print(f"Outputs will be saved to: {OUTPUT_DIR}")

## 3. Locate Existing FlowPywr Results

The notebook searches the existing result folders for the latest run matching each basin and scenario. It does not overwrite the original simulation results.

In [ ]:
BASINS = {
    "merced": {
        "label": "Merced",
        "data_folder": "Merced_River",
        "terminal_reservoir": "Lake McClure",
        "hydropower_nodes": ["New Exchequer PH", "Merced Falls PH", "McSwain PH"],
        "delivery_nodes": ["MID Main", "MID Northside"],
        "ifr_nodes": ["IFR bl New Exchequer Dam", "IFR at Shaffer Bridge"],
        "basin_outflow": "Merced River Outflow",
        "flood_release_nodes": ["Exchequer Dam Flood Release"],
    },
    "tuolumne": {
        "label": "Tuolumne",
        "data_folder": "Tuolumne_River",
        "terminal_reservoir": "Don Pedro Reservoir",
        "hydropower_nodes": ["Moccasin PH", "Kirkwood PH", "Don Pedro PH", "Dion R Holm PH"],
        "delivery_nodes": ["Modesto Irrigation District", "Turlock Irrigation District"],
        "ifr_nodes": [
            "IFR bl Hetch Hetchy Reservoir",
            "IFR bl Lake Eleanor",
            "IFR bl Cherry Lake",
            "IFR at La Grange",
        ],
        "basin_outflow": "Tuolumne River Outflow",
        "flood_release_nodes": ["Don Pedro Lake Flood Control"],
    },
}

SCENARIOS = {
    "Historical Livneh": {
        "result_glob": "Natural Flow*",
        "climate_set": "historical",
        "climate_model": "Livneh",
        "scenario_column": "Baseline",
    },
    "GCM ACCESS1-0 RCP8.5": {
        "result_glob": "GCMs test*",
        "climate_set": "gcms",
        "climate_model": "ACCESS1_0_rcp85",
        "scenario_column": None,
    },
    "LOCA2 ACCESS-CM2": {
        "result_glob": "LOCA2_GCMs test*",
        "climate_set": "LOCA2_gcms",
        "climate_model": "ACCESS_CM2",
        "scenario_column": None,
    },
}

FILES = {
    "storage": "Reservoir_Storage_mcm.csv",
    "hydropower_energy": "Hydropower_Energy_MWh.csv",
    "ifr_flow": "InstreamFlowRequirement_Flow_mcm.csv",
    "ifr_min": "InstreamFlowRequirement_Min Flow_mcm.csv",
    "output_flow": "Output_Flow_mcm.csv",
    "output_demand": "Output_Demand_mcm.csv",
    "flood_release": "PiecewiseLink_Flow_mcm.csv",
}

missing_sources = []


def latest_result_path(basin_key, scenario_label):
    scenario = SCENARIOS[scenario_label]
    pattern = f"{scenario['result_glob']}/{basin_key}/{scenario['climate_set']}/{scenario['climate_model']}"
    matches = sorted(RESULTS_ROOT.glob(pattern), key=lambda p: p.stat().st_mtime)
    if not matches:
        raise FileNotFoundError(f"No result folder found for {basin_key} / {scenario_label}. Pattern: {pattern}")
    return matches[-1]


RUNS = {}
for basin_key, basin_cfg in BASINS.items():
    for scenario_label, scenario_cfg in SCENARIOS.items():
        hydrology_path = (
            PROJECT_ROOT
            / "data"
            / basin_cfg["data_folder"]
            / "hydrology"
            / scenario_cfg["climate_set"]
            / scenario_cfg["climate_model"]
            / "preprocessed"
            / "full_natural_flow_daily_mcm.csv"
        )
        if not hydrology_path.exists():
            missing_sources.append(f"Missing hydrology file: {hydrology_path}")
        RUNS[(basin_key, scenario_label)] = {
            "path": latest_result_path(basin_key, scenario_label),
            "hydrology_path": hydrology_path,
            "scenario_column": scenario_cfg["scenario_column"],
        }

for (basin_key, scenario_label), cfg in RUNS.items():
    print(f"{BASINS[basin_key]['label']:9s} | {scenario_label:24s} | {cfg['path']}")

if missing_sources:
    print("\nMissing source diagnostics:")
    for item in missing_sources:
        print(" -", item)

## 4. Basin and Scenario Configuration

The basin configuration defines the nodes used for each Layer B metric. These are the same nodes used in the current climate-robustness analysis, with explicit basin-specific lists to avoid accidentally including unrelated recorder columns.

In [ ]:
for basin_key, cfg in BASINS.items():
    print(f"\n{cfg['label']}")
    print("  Terminal reservoir:", cfg["terminal_reservoir"])
    print("  Hydropower nodes:", ", ".join(cfg["hydropower_nodes"]))
    print("  Delivery nodes:", ", ".join(cfg["delivery_nodes"]))
    print("  IFR nodes:", ", ".join(cfg["ifr_nodes"]))
    print("  Basin outlet:", cfg["basin_outflow"])
    print("  Flood-release proxy nodes:", ", ".join(cfg["flood_release_nodes"]))

## 5. Water-Year Utilities

A water year runs from October 1 through September 30. For example, October 1, 2045 through September 30, 2046 is Water Year 2046. Leap years are handled from the actual calendar dates.

In [ ]:
def water_year(index):
    idx = pd.DatetimeIndex(index)
    return idx.year + (idx.month >= 10).astype(int)


def water_year_day(index):
    idx = pd.DatetimeIndex(index)
    wy_start_year = idx.year - (idx.month < 10).astype(int)
    starts = pd.to_datetime(pd.Series(wy_start_year, index=idx).astype(str) + "-10-01").to_numpy()
    return (idx - starts).days + 1


def expected_days_in_water_year(wy):
    start = pd.Timestamp(year=int(wy) - 1, month=10, day=1)
    end = pd.Timestamp(year=int(wy), month=9, day=30)
    return (end - start).days + 1


def check_water_year_completeness(df, label):
    wy = pd.Series(water_year(df.index), index=df.index, name="water_year")
    counts = wy.groupby(wy).size()
    expected = counts.index.to_series().apply(expected_days_in_water_year)
    complete = counts.eq(expected.values)
    report = pd.DataFrame({
        "water_year": counts.index.astype(int),
        "observed_days": counts.values,
        "expected_days": expected.values,
        "complete": complete.values,
    })
    incomplete = report.loc[~report["complete"]]
    if len(incomplete):
        print(f"WARNING: incomplete water years in {label}")
        display(incomplete)
    return report


def add_water_year(df):
    out = df.copy()
    out["water_year"] = water_year(out.index)
    return out


def annual_sum(df):
    return add_water_year(df).groupby("water_year").sum(numeric_only=True)


def annual_mean(df):
    return add_water_year(df).groupby("water_year").mean(numeric_only=True)


def annual_min(df):
    return add_water_year(df).groupby("water_year").min(numeric_only=True)


def savefig(fig, name):
    path = FIG_DIR / name
    fig.savefig(path, bbox_inches="tight")
    print(f"Saved: {path}")
    return path

## 6. Layer A: Hydrologic Stress Metrics

Layer A metrics describe the forcing and starting state. They are not operating-rule performance metrics. They are used to match future water years to comparable historical years.

In [ ]:
def read_pywr_csv(path, scenario=None):
    """Read flat Pywr CSVs and scenario-expanded Pywr CSVs."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    with path.open("r", encoding="utf-8-sig") as f:
        first_line = f.readline().strip()

    if first_line.startswith("node,"):
        df = pd.read_csv(path, header=[0, 1, 2], index_col=0)
        df.index = pd.to_datetime(df.index, errors="coerce")
        df = df.loc[df.index.notna()].copy()
        df.index.name = "date"
        df.columns = pd.MultiIndex.from_tuples(
            [(str(a), str(b), str(c)) for a, b, c in df.columns],
            names=["node", "scenario", "unused"],
        )
        available = sorted(set(df.columns.get_level_values("scenario")))
        if scenario is None:
            scenario = available[0]
        if scenario not in available:
            raise ValueError(f"Scenario {scenario!r} not found in {path.name}. Available: {available}")
        df = df.xs(scenario, level="scenario", axis=1, drop_level=False)
        df.columns = df.columns.get_level_values("node")
    else:
        df = pd.read_csv(path, index_col=0)
        df.index = pd.to_datetime(df.index, errors="coerce")
        df = df.loc[df.index.notna()].copy()
        df.index.name = "date"

    df = df.apply(pd.to_numeric, errors="coerce").sort_index()
    return df


def read_run_file(basin_key, scenario_label, file_key):
    cfg = RUNS[(basin_key, scenario_label)]
    return read_pywr_csv(cfg["path"] / FILES[file_key], scenario=cfg["scenario_column"])


def read_hydrology(basin_key, scenario_label):
    cfg = RUNS[(basin_key, scenario_label)]
    df = pd.read_csv(cfg["hydrology_path"], index_col=0)
    df.index = pd.to_datetime(df.index, errors="coerce")
    df = df.loc[df.index.notna()].copy()
    df.index.name = "date"
    if "flow" not in df.columns:
        df.columns = ["flow"]
    df["flow"] = pd.to_numeric(df["flow"], errors="coerce")
    return df[["flow"]].sort_index()


def require_columns(df, columns, label):
    missing = [c for c in columns if c not in df.columns]
    if missing:
        message = f"Missing recorder columns in {label}: {missing}. Available columns: {list(df.columns)}"
        missing_sources.append(message)
        raise KeyError(message)
    return df[columns]


def align_to_model_period(hydrology, reference_df, label):
    start, end = reference_df.index.min(), reference_df.index.max()
    aligned = hydrology.loc[(hydrology.index >= start) & (hydrology.index <= end)].copy()
    if aligned.empty:
        raise ValueError(f"No hydrology data overlaps model output dates for {label}: {start} to {end}")
    missing_dates = pd.DatetimeIndex(reference_df.index).difference(aligned.index)
    if len(missing_dates):
        print(f"WARNING: {label} hydrology is missing {len(missing_dates)} model-output dates.")
    return aligned.reindex(reference_df.index)


def calculate_layer_a_metrics(hydrology, storage, terminal_reservoir):
    hydro = hydrology[["flow"]].copy()
    hydro["water_year"] = water_year(hydro.index)
    hydro["wy_day"] = water_year_day(hydro.index)

    annual_runoff = hydro.groupby("water_year")["flow"].sum()
    apr_jul = hydro.loc[hydro.index.month.isin([4, 5, 6, 7])].groupby("water_year")["flow"].sum()

    def center_of_mass(group):
        total = group["flow"].sum()
        if pd.isna(total) or total == 0:
            return np.nan
        return (group["wy_day"] * group["flow"]).sum() / total

    timing = hydro.groupby("water_year").apply(center_of_mass)

    st = storage[[terminal_reservoir]].copy()
    st["water_year"] = water_year(st.index)
    initial_storage = st.groupby("water_year")[terminal_reservoir].first()

    return pd.DataFrame({
        "annual_runoff_mcm": annual_runoff,
        "apr_jul_runoff_mcm": apr_jul,
        "runoff_center_of_mass_day": timing,
        "initial_storage_mcm": initial_storage,
    }).reset_index()

## 7. Layer B: Raw System Outcomes

Layer B metrics describe modeled system outcomes by water year using the same definitions as the current climate-robustness analysis.

In [ ]:
def calculate_layer_b_metrics(sections, basin_cfg, label):
    terminal = basin_cfg["terminal_reservoir"]
    hydropower_nodes = basin_cfg["hydropower_nodes"]
    delivery_nodes = basin_cfg["delivery_nodes"]
    ifr_nodes = basin_cfg["ifr_nodes"]
    outflow_node = basin_cfg["basin_outflow"]
    flood_nodes = basin_cfg["flood_release_nodes"]

    storage = require_columns(sections["storage"], [terminal], f"{label} storage")
    hydropower = require_columns(sections["hydropower_energy"], hydropower_nodes, f"{label} hydropower")
    output_flow = require_columns(sections["output_flow"], delivery_nodes + [outflow_node], f"{label} output flow")
    output_demand = require_columns(sections["output_demand"], delivery_nodes, f"{label} output demand")
    ifr_flow = require_columns(sections["ifr_flow"], ifr_nodes, f"{label} IFR flow")
    ifr_min = require_columns(sections["ifr_min"], ifr_nodes, f"{label} IFR min flow")

    flood_release = None
    if "flood_release" in sections:
        available_flood = [node for node in flood_nodes if node in sections["flood_release"].columns]
        if available_flood:
            flood_release = sections["flood_release"][available_flood]
        else:
            missing_sources.append(f"No configured flood-release proxy nodes found for {label}. Available: {list(sections['flood_release'].columns)}")
    else:
        missing_sources.append(f"Missing flood-release CSV for {label}")

    delivery = output_flow[delivery_nodes]
    demand = output_demand[delivery_nodes]
    delivery_shortage = (demand - delivery).clip(lower=0)

    ifr_deficit = (ifr_min - ifr_flow).clip(lower=0)

    annual_demand = annual_sum(demand).sum(axis=1)
    annual_shortage = annual_sum(delivery_shortage).sum(axis=1)
    annual_ifr_req = annual_sum(ifr_min).sum(axis=1)
    annual_ifr_deficit = annual_sum(ifr_deficit).sum(axis=1)

    annual = pd.DataFrame({
        "mean_terminal_storage_mcm": annual_mean(storage)[terminal],
        "minimum_terminal_storage_mcm": annual_min(storage)[terminal],
        "annual_hydropower_generation": annual_sum(hydropower).sum(axis=1),
        "annual_delivery_demand_mcm": annual_demand,
        "annual_delivery_shortage_mcm": annual_shortage,
        "delivery_reliability": 1 - annual_shortage.divide(annual_demand.replace(0, np.nan)),
        "annual_ifr_requirement_mcm": annual_ifr_req,
        "annual_ifr_deficit_mcm": annual_ifr_deficit,
        "ifr_reliability": 1 - annual_ifr_deficit.divide(annual_ifr_req.replace(0, np.nan)),
        "annual_basin_outflow_mcm": annual_sum(output_flow[[outflow_node]])[outflow_node],
        "annual_flood_release_proxy_mcm": annual_sum(flood_release).sum(axis=1) if flood_release is not None else np.nan,
    }).reset_index()

    return annual

## 8. Combined Annual Metrics Table

This section creates one master table with one row per basin, scenario, and water year. It contains Layer A hydrologic stress metrics and Layer B raw system outcomes.

In [ ]:
data = {basin_key: {scenario_label: {} for scenario_label in SCENARIOS} for basin_key in BASINS}
annual_rows = []
completeness_reports = []
daily_runoff_rows = []

for basin_key, basin_cfg in BASINS.items():
    for scenario_label in SCENARIOS:
        label = f"{basin_cfg['label']} / {scenario_label}"
        sections = data[basin_key][scenario_label]

        for file_key in FILES:
            path = RUNS[(basin_key, scenario_label)]["path"] / FILES[file_key]
            if path.exists():
                sections[file_key] = read_run_file(basin_key, scenario_label, file_key)
            else:
                missing_sources.append(f"Missing result file for {label}: {path}")

        required_keys = ["storage", "hydropower_energy", "ifr_flow", "ifr_min", "output_flow", "output_demand"]
        missing_keys = [key for key in required_keys if key not in sections]
        if missing_keys:
            raise FileNotFoundError(f"Missing required result sections for {label}: {missing_keys}")

        sections["hydrology_daily"] = align_to_model_period(
            read_hydrology(basin_key, scenario_label),
            sections["storage"],
            label,
        )

        daily_runoff = sections["hydrology_daily"].reset_index().rename(columns={"flow": "daily_runoff_mcm"})
        daily_runoff.insert(0, "scenario", scenario_label)
        daily_runoff.insert(0, "basin_label", basin_cfg["label"])
        daily_runoff.insert(0, "basin", basin_key)
        daily_runoff["water_year"] = water_year(daily_runoff["date"])
        daily_runoff["water_year_day"] = water_year_day(daily_runoff["date"])
        daily_runoff_rows.append(daily_runoff)

        completeness = check_water_year_completeness(sections["storage"], label)
        completeness["basin"] = basin_key
        completeness["scenario"] = scenario_label
        completeness_reports.append(completeness)

        layer_a = calculate_layer_a_metrics(
            sections["hydrology_daily"],
            sections["storage"],
            basin_cfg["terminal_reservoir"],
        )
        layer_b = calculate_layer_b_metrics(sections, basin_cfg, label)

        annual = layer_a.merge(layer_b, on="water_year", how="outer", validate="one_to_one")
        annual.insert(0, "scenario", scenario_label)
        annual.insert(0, "basin", basin_key)
        annual.insert(1, "basin_label", basin_cfg["label"])
        annual_rows.append(annual)

annual_metrics = pd.concat(annual_rows, ignore_index=True).sort_values(["basin", "scenario", "water_year"])
daily_runoff = pd.concat(daily_runoff_rows, ignore_index=True).sort_values(["basin", "scenario", "date"])
water_year_completeness = pd.concat(completeness_reports, ignore_index=True)

print("annual_metrics.head()")
display(annual_metrics.head())
print("annual_metrics.shape", annual_metrics.shape)
print("\nRows by basin/scenario:")
display(annual_metrics.groupby(["basin", "scenario"]).size().rename("n_water_years"))

missing_by_col = annual_metrics.isna().sum().loc[lambda s: s > 0]
if len(missing_by_col):
    print("\nMissing values by column:")
    display(missing_by_col)
else:
    print("\nNo missing values found in annual_metrics.")

if missing_sources:
    print("\nSource/recorder diagnostics:")
    for item in missing_sources:
        print(" -", item)
else:
    print("\nNo missing source files or configured recorder outputs were detected.")

## 9. Layer C: Hydrology-Conditioned Matching

For each basin, Historical Livneh water years are the reference pool. For every future GCM or LOCA2 water year, the notebook finds the three historical years with the smallest standardized distance using annual runoff, April-July runoff, runoff center-of-mass day, and initial terminal reservoir storage.

The hydrology-conditioned performance difference is calculated as:

```text
future scenario value - mean value of matched historical years
```

For lower-is-better metrics such as shortage and deficit, the sign is not reversed. Positive values still mean the future year is higher than its matched historical expectation.

In [ ]:
HYDROLOGY_MATCH_FEATURES = [
    "annual_runoff_mcm",
    "apr_jul_runoff_mcm",
    "runoff_center_of_mass_day",
    "initial_storage_mcm",
]

OUTCOME_METRICS = [
    "mean_terminal_storage_mcm",
    "minimum_terminal_storage_mcm",
    "annual_hydropower_generation",
    "annual_delivery_demand_mcm",
    "annual_delivery_shortage_mcm",
    "delivery_reliability",
    "annual_ifr_requirement_mcm",
    "annual_ifr_deficit_mcm",
    "ifr_reliability",
    "annual_basin_outflow_mcm",
    "annual_flood_release_proxy_mcm",
]


def build_hydrology_matches(annual_metrics, k=3):
    rows = []
    for basin_key, basin_group in annual_metrics.groupby("basin"):
        historical = basin_group.loc[basin_group["scenario"] == HISTORICAL_SCENARIO].copy()
        future = basin_group.loc[basin_group["scenario"].isin(FUTURE_SCENARIOS)].copy()

        if len(historical) == 0:
            raise ValueError(f"No historical reference years found for basin {basin_key}")
        if len(future) == 0:
            continue

        feature_na = historical[HYDROLOGY_MATCH_FEATURES].isna().sum()
        if feature_na.any():
            raise ValueError(f"Historical matching features contain missing values for {basin_key}: {feature_na[feature_na > 0].to_dict()}")

        k_eff = min(k, len(historical))
        scaler = StandardScaler()
        hist_x = scaler.fit_transform(historical[HYDROLOGY_MATCH_FEATURES])
        future_x = scaler.transform(future[HYDROLOGY_MATCH_FEATURES])

        nn = NearestNeighbors(n_neighbors=k_eff, metric="euclidean")
        nn.fit(hist_x)
        distances, indices = nn.kneighbors(future_x)
        historical_indexed = historical.reset_index(drop=True)

        for future_pos, (_, future_row) in enumerate(future.iterrows()):
            match_rows = historical_indexed.iloc[indices[future_pos]].copy()
            match_distances = distances[future_pos]

            out = {
                "basin": basin_key,
                "basin_label": future_row["basin_label"],
                "scenario": future_row["scenario"],
                "water_year": int(future_row["water_year"]),
                **{col: future_row[col] for col in HYDROLOGY_MATCH_FEATURES},
                "mean_match_distance": float(np.mean(match_distances)),
            }

            for i in range(k):
                if i < k_eff:
                    out[f"matched_historical_year_{i + 1}"] = int(match_rows.iloc[i]["water_year"])
                    out[f"match_distance_{i + 1}"] = float(match_distances[i])
                else:
                    out[f"matched_historical_year_{i + 1}"] = np.nan
                    out[f"match_distance_{i + 1}"] = np.nan

            for metric in OUTCOME_METRICS:
                matched_mean = match_rows[metric].mean()
                actual = future_row[metric]
                out[f"actual_{metric}"] = actual
                out[f"matched_historical_{metric}"] = matched_mean
                out[f"difference_{metric}"] = actual - matched_mean

            rows.append(out)

    matches = pd.DataFrame(rows)

    thresholds = (
        matches.groupby("basin")["mean_match_distance"]
        .quantile([0.75, 0.90, 0.95])
        .unstack()
        .rename(columns={0.75: "distance_p75", 0.90: "distance_p90", 0.95: "distance_p95"})
        .reset_index()
    )
    matches = matches.merge(thresholds, on="basin", how="left")
    matches["match_quality"] = np.where(
        matches["mean_match_distance"] > matches["distance_p90"],
        "weak historical analog",
        "acceptable analog",
    )
    return matches, thresholds


hydrology_matches, match_distance_thresholds = build_hydrology_matches(annual_metrics, k=K_MATCHES)

print("hydrology_matches.head()")
display(hydrology_matches.head())
print("hydrology_matches.shape", hydrology_matches.shape)
print("\nMatch distance thresholds by basin:")
display(match_distance_thresholds.round(3))
print("\nWeak historical analog years:")
display(hydrology_matches.loc[hydrology_matches["match_quality"] == "weak historical analog", [
    "basin_label", "scenario", "water_year", "mean_match_distance", "distance_p90",
    "annual_runoff_mcm", "apr_jul_runoff_mcm", "runoff_center_of_mass_day", "initial_storage_mcm",
]])

## 10. Match Quality Diagnostics

Nearest-neighbor matching always returns a match, but a returned match is not necessarily a close historical analog. This section reports the empirical distribution of mean match distances and flags years above the basin-specific 90th percentile as weak historical analogs. These years are retained in all calculations.

In [ ]:
def safe_filename(text):
    return (
        text.lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace(".", "")
        .replace("-", "_")
    )


def plot_future_vs_analogs(matches, basin_key, scenario_label):
    """Plot match distance from each future water year to its three historical analogs."""
    subset = matches.loc[
        (matches["basin"] == basin_key) &
        (matches["scenario"] == scenario_label)
    ].copy().sort_values("water_year")

    if subset.empty:
        print(f"No data found for {basin_key} / {scenario_label}")
        return None

    fig, ax = plt.subplots(figsize=(12, 5))
    scenario_color = PALETTE.get(scenario_label, "#333333")
    analog_cols = [c for c in ["match_distance_1", "match_distance_2", "match_distance_3"] if c in subset.columns]

    for i, col in enumerate(analog_cols, start=1):
        ax.plot(
            subset["water_year"],
            subset[col],
            marker="o",
            linewidth=1.0,
            markersize=3.8,
            alpha=0.38,
            color="0.45",
            label=f"Historical analog {i}",
        )

    ax.plot(
        subset["water_year"],
        subset["mean_match_distance"],
        marker="o",
        linewidth=2.6,
        markersize=6,
        color=scenario_color,
        label="Average of 3 historical analogs",
    )

    ax.axhline(
        subset["distance_p90"].iloc[0],
        color="black",
        linestyle="--",
        linewidth=1.2,
        label="90th percentile threshold",
    )

    weak = subset["match_quality"].eq("weak historical analog")
    if weak.any():
        ax.scatter(
            subset.loc[weak, "water_year"],
            subset.loc[weak, "mean_match_distance"],
            s=115,
            facecolor="none",
            edgecolor="black",
            linewidth=1.5,
            zorder=5,
            label="Weak historical analog",
        )

    ax.set_title(
        f"{BASINS[basin_key]['label']} - {scenario_label}\n"
        "Distance from Future Water Years to Matched Historical Analogs"
    )
    ax.set_xlabel("Future water year")
    ax.set_ylabel("Standardized hydrologic distance")
    ax.grid(True, axis="y", alpha=0.25)
    ax.legend(frameon=False, ncol=2)

    fig.tight_layout()
    filename = f"figB_detail_match_distance_{basin_key}_{safe_filename(scenario_label)}.png"
    savefig(fig, filename)
    plt.show()
    return fig


# Four separate plots: two future scenarios for each basin.
for basin_key in BASINS:
    for scenario_label in FUTURE_SCENARIOS:
        plot_future_vs_analogs(hydrology_matches, basin_key, scenario_label)


In [ ]:
def plot_match_quality(matches):
    basins = list(BASINS.keys())
    fig, axes = plt.subplots(len(basins), 1, figsize=(12, 4.0 * len(basins)), sharex=False)
    if len(basins) == 1:
        axes = [axes]

    for ax, basin_key in zip(axes, basins):
        subset = matches.loc[matches["basin"] == basin_key].copy()
        for scenario_label in FUTURE_SCENARIOS:
            s = subset.loc[subset["scenario"] == scenario_label].sort_values("water_year")
            ax.plot(
                s["water_year"],
                s["mean_match_distance"],
                color=PALETTE[scenario_label],
                marker="o",
                linewidth=1.8,
                markersize=4.5,
                label=scenario_label,
            )
            weak = s["match_quality"].eq("weak historical analog")
            ax.scatter(
                s.loc[weak, "water_year"],
                s.loc[weak, "mean_match_distance"],
                s=90,
                facecolor="none",
                edgecolor="black",
                linewidth=1.4,
                zorder=4,
            )

        ax.axhline(subset["distance_p90"].iloc[0], color="black", linestyle="--", linewidth=1.0, label="90th percentile")
        ax.set_title(f"{BASINS[basin_key]['label']} Match Quality")
        ax.set_xlabel("Water year")
        ax.set_ylabel("Mean standardized distance")
        ax.grid(True, axis="y", alpha=0.25)

    handles, labels = axes[0].get_legend_handles_labels()
    weak_handle = plt.Line2D([0], [0], marker='o', color='black', markerfacecolor='none', label='Weak historical analog', markersize=8, linewidth=0)
    handles.append(weak_handle)
    labels.append('Weak historical analog')
    fig.legend(handles, labels, loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 1.03))
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    savefig(fig, "figB_match_quality_by_water_year.png")
    plt.show()


plot_match_quality(hydrology_matches)

display(hydrology_matches.groupby(["basin_label", "scenario", "match_quality"]).size().rename("n_years").reset_index())

## 11. Hydrology-Conditioned Performance Comparisons

This section compares each future water year to the mean outcome from its matched historical years. A small hydrology-conditioned difference suggests that raw performance differences are largely consistent with the selected hydrologic conditions. A large difference identifies behavior that is less easily explained by annual runoff, spring runoff, runoff timing, and initial storage alone.

In [ ]:
DISPLAY_METRICS = {
    "mean_terminal_storage_mcm": {"label": "Mean terminal storage", "unit": "mcm", "scale": 1.0},
    "annual_hydropower_generation": {"label": "Hydropower generation", "unit": "MWh/year", "scale": 1.0},
    "delivery_reliability": {"label": "Delivery reliability", "unit": "%", "scale": 100.0},
    "annual_delivery_shortage_mcm": {"label": "Delivery shortage", "unit": "mcm/year", "scale": 1.0},
    "ifr_reliability": {"label": "IFR reliability", "unit": "%", "scale": 100.0},
    "annual_ifr_deficit_mcm": {"label": "IFR deficit", "unit": "mcm/year", "scale": 1.0},
    "annual_basin_outflow_mcm": {"label": "Basin outflow", "unit": "mcm/year", "scale": 1.0},
}


def scenario_short_name(label):
    if label.startswith("GCM"):
        return "GCM"
    if label.startswith("LOCA2"):
        return "LOCA2"
    if label.startswith("Historical"):
        return "Historical"
    return label


def grouped_future_labels(matches):
    combos = matches[["basin", "basin_label", "scenario"]].drop_duplicates().sort_values(["basin", "scenario"])
    labels = [f"{row.basin_label}\n{scenario_short_name(row.scenario)}" for row in combos.itertuples()]
    return combos, labels


def plot_raw_vs_matched(matches, metric_names, filename, title):
    combos, labels = grouped_future_labels(matches)
    n_metrics = len(metric_names)
    ncols = 2 if n_metrics > 1 else 1
    nrows = int(np.ceil(n_metrics / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(12.5, 4.2 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for ax, metric in zip(axes, metric_names):
        meta = DISPLAY_METRICS[metric]
        actual_means = []
        matched_means = []
        colors = []
        for row in combos.itertuples():
            subset = matches.loc[(matches["basin"] == row.basin) & (matches["scenario"] == row.scenario)]
            actual_means.append(subset[f"actual_{metric}"].mean() * meta["scale"])
            matched_means.append(subset[f"matched_historical_{metric}"].mean() * meta["scale"])
            colors.append(PALETTE[row.scenario])

        x = np.arange(len(labels))
        width = 0.36
        ax.bar(x - width / 2, actual_means, width=width, color=colors, alpha=0.82, edgecolor="black", linewidth=0.4, label="Future actual")
        ax.bar(x + width / 2, matched_means, width=width, color="white", edgecolor=colors, hatch="///", linewidth=1.2, label="Matched historical mean")
        ax.set_title(meta["label"])
        ax.set_ylabel(meta["unit"])
        ax.set_xticks(x)
        ax.set_xticklabels(labels)
        ax.grid(True, axis="y", alpha=0.25)

    for ax in axes[n_metrics:]:
        ax.axis("off")

    handles, legend_labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, legend_labels, loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.02))
    fig.suptitle(title, y=1.04, fontsize=14, fontweight="bold")
    fig.tight_layout()
    savefig(fig, filename)
    plt.show()


plot_raw_vs_matched(
    hydrology_matches,
    ["mean_terminal_storage_mcm", "annual_hydropower_generation", "annual_basin_outflow_mcm"],
    "figC1_raw_vs_matched_storage_hydropower_outflow.png",
    "Raw Future Outcomes vs Matched Historical Expectations",
)

plot_raw_vs_matched(
    hydrology_matches,
    ["delivery_reliability", "annual_delivery_shortage_mcm", "ifr_reliability", "annual_ifr_deficit_mcm"],
    "figC2_raw_vs_matched_reliability_shortage_deficit.png",
    "Raw Future Outcomes vs Matched Historical Expectations",
)

## 12. Figures

The figures below summarize the hydrologic stress space and hydrology-conditioned performance differences. They are saved to the analysis output directory.

In [ ]:
def plot_hydrologic_stress_space(df):
    basins = list(BASINS.keys())
    fig, axes = plt.subplots(1, len(basins), figsize=(13, 5), sharey=False)
    axes = np.atleast_1d(axes)

    for ax, basin_key in zip(axes, basins):
        subset = df.loc[df["basin"] == basin_key]
        for scenario_label in SCENARIOS:
            s = subset.loc[subset["scenario"] == scenario_label]
            ax.scatter(
                s["annual_runoff_mcm"],
                s["runoff_center_of_mass_day"],
                label=scenario_label,
                color=PALETTE[scenario_label],
                marker=MARKERS[scenario_label],
                s=45,
                alpha=0.78,
                edgecolor="black",
                linewidth=0.3,
            )
        ax.set_title(f"{BASINS[basin_key]['label']} Hydrologic Stress Space")
        ax.set_xlabel("Annual runoff (mcm/year)")
        ax.set_ylabel("Runoff center-of-mass day")
        ax.grid(True, alpha=0.25)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.05))
    fig.tight_layout()
    savefig(fig, "figA1_hydrologic_stress_annual_runoff_vs_timing.png")
    plt.show()


def plot_runoff_partition_space(df):
    basins = list(BASINS.keys())
    fig, axes = plt.subplots(1, len(basins), figsize=(13, 5), sharey=False)
    axes = np.atleast_1d(axes)

    for ax, basin_key in zip(axes, basins):
        subset = df.loc[df["basin"] == basin_key]
        for scenario_label in SCENARIOS:
            s = subset.loc[subset["scenario"] == scenario_label]
            ax.scatter(
                s["annual_runoff_mcm"],
                s["apr_jul_runoff_mcm"],
                label=scenario_label,
                color=PALETTE[scenario_label],
                marker=MARKERS[scenario_label],
                s=45,
                alpha=0.78,
                edgecolor="black",
                linewidth=0.3,
            )
        ax.set_title(f"{BASINS[basin_key]['label']} Annual vs Apr-Jul Runoff")
        ax.set_xlabel("Annual runoff (mcm/year)")
        ax.set_ylabel("Apr-Jul runoff (mcm/year)")
        ax.grid(True, alpha=0.25)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.05))
    fig.tight_layout()
    savefig(fig, "figA2_hydrologic_stress_annual_vs_apr_jul_runoff.png")
    plt.show()


plot_hydrologic_stress_space(annual_metrics)
plot_runoff_partition_space(annual_metrics)

In [ ]:
def plot_difference_boxplots(matches, metric_names, filename, title):
    combos, labels = grouped_future_labels(matches)
    n_metrics = len(metric_names)
    ncols = 2 if n_metrics > 1 else 1
    nrows = int(np.ceil(n_metrics / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(12.5, 4.2 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for ax, metric in zip(axes, metric_names):
        meta = DISPLAY_METRICS[metric]
        series = []
        colors = []
        for row in combos.itertuples():
            subset = matches.loc[(matches["basin"] == row.basin) & (matches["scenario"] == row.scenario)]
            values = subset[f"difference_{metric}"].dropna().to_numpy() * meta["scale"]
            series.append(values)
            colors.append(PALETTE[row.scenario])

        bp = ax.boxplot(series, patch_artist=True, labels=labels, showfliers=False)
        for patch, color in zip(bp["boxes"], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.68)
            patch.set_edgecolor("black")
        for median in bp["medians"]:
            median.set_color("black")
            median.set_linewidth(1.2)
        for i, values in enumerate(series, start=1):
            jitter = np.linspace(-0.08, 0.08, len(values)) if len(values) else []
            ax.scatter(np.full(len(values), i) + jitter, values, color=colors[i - 1], alpha=0.35, s=16, edgecolor="none")
        ax.axhline(0, color="black", linewidth=1, linestyle="--")
        ax.set_title(meta["label"])
        ax.set_ylabel(f"Future - matched historical ({meta['unit']})")
        ax.grid(True, axis="y", alpha=0.25)

    for ax in axes[n_metrics:]:
        ax.axis("off")

    fig.suptitle(title, y=1.02, fontsize=14, fontweight="bold")
    fig.tight_layout()
    savefig(fig, filename)
    plt.show()


plot_difference_boxplots(
    hydrology_matches,
    ["mean_terminal_storage_mcm", "annual_hydropower_generation", "annual_basin_outflow_mcm"],
    "figD1_hydrology_conditioned_differences_storage_hydropower_outflow.png",
    "Hydrology-Conditioned Performance Differences",
)

plot_difference_boxplots(
    hydrology_matches,
    ["delivery_reliability", "annual_delivery_shortage_mcm", "ifr_reliability", "annual_ifr_deficit_mcm"],
    "figD2_hydrology_conditioned_differences_reliability_shortage_deficit.png",
    "Hydrology-Conditioned Performance Differences",
)

## 13. Summary Tables

The summary table reports hydrologic stress means, match quality, raw future mean performance, matched historical mean performance, and hydrology-conditioned differences for each basin and future scenario.

In [ ]:
def build_conditioned_summary(matches):
    rows = []
    for (basin, scenario), group in matches.groupby(["basin", "scenario"]):
        out = {
            "basin": basin,
            "basin_label": group["basin_label"].iloc[0],
            "scenario": scenario,
            "n_water_years": len(group),
            "mean_annual_runoff_mcm": group["annual_runoff_mcm"].mean(),
            "mean_apr_jul_runoff_mcm": group["apr_jul_runoff_mcm"].mean(),
            "mean_runoff_center_of_mass_day": group["runoff_center_of_mass_day"].mean(),
            "mean_initial_storage_mcm": group["initial_storage_mcm"].mean(),
            "mean_nearest_neighbor_distance": group["mean_match_distance"].mean(),
            "weak_historical_analog_years": int((group["match_quality"] == "weak historical analog").sum()),
        }
        for metric in OUTCOME_METRICS:
            out[f"raw_mean_{metric}"] = group[f"actual_{metric}"].mean()
            out[f"matched_historical_mean_{metric}"] = group[f"matched_historical_{metric}"].mean()
            out[f"mean_hydrology_conditioned_difference_{metric}"] = group[f"difference_{metric}"].mean()
            out[f"median_hydrology_conditioned_difference_{metric}"] = group[f"difference_{metric}"].median()
        rows.append(out)
    return pd.DataFrame(rows).sort_values(["basin", "scenario"])


hydrology_conditioned_summary = build_conditioned_summary(hydrology_matches)

concise_columns = [
    "basin_label", "scenario", "n_water_years", "mean_annual_runoff_mcm", "mean_apr_jul_runoff_mcm",
    "mean_runoff_center_of_mass_day", "mean_initial_storage_mcm", "mean_nearest_neighbor_distance",
    "weak_historical_analog_years",
    "raw_mean_delivery_reliability", "matched_historical_mean_delivery_reliability",
    "mean_hydrology_conditioned_difference_delivery_reliability",
    "raw_mean_annual_delivery_shortage_mcm", "matched_historical_mean_annual_delivery_shortage_mcm",
    "mean_hydrology_conditioned_difference_annual_delivery_shortage_mcm",
    "raw_mean_ifr_reliability", "matched_historical_mean_ifr_reliability",
    "mean_hydrology_conditioned_difference_ifr_reliability",
    "raw_mean_annual_ifr_deficit_mcm", "matched_historical_mean_annual_ifr_deficit_mcm",
    "mean_hydrology_conditioned_difference_annual_ifr_deficit_mcm",
]
concise_summary = hydrology_conditioned_summary[concise_columns].copy()

print("Hydrology-conditioned summary:")
display(hydrology_conditioned_summary.round(3))
print("\nConcise summary for Merced GCM, Merced LOCA2, Tuolumne GCM, and Tuolumne LOCA2:")
display(concise_summary.round(3))

## 14. Key Diagnostic Findings

This section prints conservative diagnostics from the computed tables. These are not proof of operating-rule inefficiency; they identify where future performance differs from the matched historical expectation and therefore deserves closer investigation.

In [ ]:
def summarize_diagnostics(summary):
    print("Layer A tells us the hydrologic and initial-storage conditions for each water year.")
    print("Layer B tells us the raw model outcomes under those conditions.")
    print("Layer C compares future outcomes to historical Livneh years with similar hydrologic stress.")
    print("\nLargest mean hydrology-conditioned differences by selected metric:")

    selected = [
        "mean_terminal_storage_mcm",
        "annual_hydropower_generation",
        "delivery_reliability",
        "annual_delivery_shortage_mcm",
        "ifr_reliability",
        "annual_ifr_deficit_mcm",
        "annual_basin_outflow_mcm",
    ]
    for metric in selected:
        col = f"mean_hydrology_conditioned_difference_{metric}"
        if col not in summary:
            continue
        temp = summary[["basin_label", "scenario", col]].copy()
        temp["abs_difference"] = temp[col].abs()
        row = temp.sort_values("abs_difference", ascending=False).iloc[0]
        value = row[col]
        print(f" - {metric}: largest absolute mean difference is {value:.3f} for {row['basin_label']} / {row['scenario']}")

    print("\nWeak analog counts:")
    display(summary[["basin_label", "scenario", "weak_historical_analog_years", "n_water_years", "mean_nearest_neighbor_distance"]].round(3))


summarize_diagnostics(hydrology_conditioned_summary)

## 15. Saved Outputs

The notebook saves the master annual metrics table, the matched-hydrology table, the hydrology-conditioned summary table, a concise summary table, and all generated figures. Original model simulation results are not overwritten.

In [ ]:
annual_metrics_path = OUTPUT_DIR / "annual_metrics.csv"
hydrology_matches_path = OUTPUT_DIR / "hydrology_matches.csv"
hydrology_conditioned_summary_path = OUTPUT_DIR / "hydrology_conditioned_summary.csv"
concise_summary_path = OUTPUT_DIR / "hydrology_conditioned_concise_summary.csv"
water_year_completeness_path = OUTPUT_DIR / "water_year_completeness.csv"
daily_runoff_path = OUTPUT_DIR / "daily_runoff.csv"

annual_metrics.to_csv(annual_metrics_path, index=False)
hydrology_matches.to_csv(hydrology_matches_path, index=False)
hydrology_conditioned_summary.to_csv(hydrology_conditioned_summary_path, index=False)
concise_summary.to_csv(concise_summary_path, index=False)
water_year_completeness.to_csv(water_year_completeness_path, index=False)
daily_runoff.to_csv(daily_runoff_path, index=False)

print("Saved outputs:")
for path in [
    annual_metrics_path,
    hydrology_matches_path,
    hydrology_conditioned_summary_path,
    concise_summary_path,
    water_year_completeness_path,
    daily_runoff_path,
]:
    print(" -", path)

print("\nSaved figures:")
for path in sorted(FIG_DIR.glob("*.png")):
    print(" -", path)

## Interpretation Notes

**Layer A** tells us what hydrology gave the system: total runoff, spring runoff, runoff timing, and initial terminal reservoir storage. These variables define the stress context for each water year.

**Layer B** tells us the raw modeled outcomes: storage, hydropower, deliveries, IFR performance, downstream outflow, and flood-release proxy. These raw outcomes are necessary, but by themselves they mix climate forcing effects with operating-rule behavior.

**Hydrology-conditioned comparison** adds a stronger diagnostic by comparing future water years only with historical Livneh years that had similar hydrologic stress. This helps separate raw climate effects from performance differences that remain after accounting for selected hydrologic conditions.

This is stronger than simply comparing scenario means because scenario means can make a dry future look worse simply because less water was available. Matching asks whether the future outcome is unusual compared with historical years that had similar water availability, runoff timing, and initial storage.

What remains unresolved is whether any hydrology-conditioned difference is caused by inefficient operating rules, changed seasonal sequences not captured by these four variables, structural model assumptions, or constraints not included in the matching space. Later extensions such as benchmark policies, optimized rules, operational regret, resilience, vulnerability, and residual models are needed before making stronger claims about rule inefficiency.